In [ ]:
import numpy as np
from numpy.random import rand, randint
import gurobipy as gp
from gurobipy import GRB
from TakeTwoSetup import *

# TODO: Turn things into functions

In [ ]:
# --- Finding subsets of scenes P --- #
#TODO: find decreasing subsets in relation of scene number

def finding_P(n, scenes, durations, W, idx, start):
    admissibleP = []

    # compute total duration of current subset
    total = sum(durations[i] for i in idx)

    # current subset is admissible
    admissibleP.append([scenes[i] for i in idx])

    for i in range(start, n):
        if total + durations[i] <= W:
            admissibleP += finding_P(n, scenes, durations, W, idx + [i], i+1)

    return admissibleP

P = finding_P(n, scenes, durations, W, [], 0)[1:]
k = len(P)
print(P)

[['Scene1'], ['Scene1', 'Scene2'], ['Scene1', 'Scene2', 'Scene7'], ['Scene1', 'Scene3'], ['Scene1', 'Scene3', 'Scene4'], ['Scene1', 'Scene3', 'Scene5'], ['Scene1', 'Scene3', 'Scene7'], ['Scene1', 'Scene3', 'Scene9'], ['Scene1', 'Scene4'], ['Scene1', 'Scene4', 'Scene5'], ['Scene1', 'Scene4', 'Scene7'], ['Scene1', 'Scene5'], ['Scene1', 'Scene5', 'Scene7'], ['Scene1', 'Scene5', 'Scene9'], ['Scene1', 'Scene6'], ['Scene1', 'Scene6', 'Scene7'], ['Scene1', 'Scene7'], ['Scene1', 'Scene7', 'Scene8'], ['Scene1', 'Scene7', 'Scene9'], ['Scene1', 'Scene8'], ['Scene1', 'Scene9'], ['Scene2'], ['Scene2', 'Scene3'], ['Scene2', 'Scene3', 'Scene4'], ['Scene2', 'Scene3', 'Scene5'], ['Scene2', 'Scene3', 'Scene7'], ['Scene2', 'Scene3', 'Scene9'], ['Scene2', 'Scene4'], ['Scene2', 'Scene4', 'Scene5'], ['Scene2', 'Scene4', 'Scene7'], ['Scene2', 'Scene5'], ['Scene2', 'Scene5', 'Scene7'], ['Scene2', 'Scene5', 'Scene9'], ['Scene2', 'Scene6'], ['Scene2', 'Scene6', 'Scene7'], ['Scene2', 'Scene7'], ['Scene2', 'Scene

In [ ]:
# --- ILP_pat Model Setup --- #
ILP_pat = gp.Model("ILP_pat", env=env)

# decision variables
a = ILP_pat.addVars(k, n,vtype=GRB.BINARY, name="a")
x_pat = ILP_pat.addVars(k, D, vtype=GRB.BINARY, name="x_pat")
y_pat = ILP_pat.addVars(m, D, D, vtype=GRB.BINARY, name="y_pat")

# objective function
objFunc_pat = ILP_pat.setObjective(
    gp.quicksum(
        holdingCost[i] *
        gp.quicksum(
              (d2-d1+1)*y_pat[i, d1, d2]
              for d1 in range(D)
              for d2 in range(d1, D)
        )
        for i in range(len(actors))
    ), GRB.MINIMIZE
)

# --- constraints (lord help us) --- #
c21 = ILP_pat.addConstr(
    gp.quicksum(
        x_pat[k, 1] for k in range(len(P))
    ) == 1, name="c21"
) # Constraint (21)

c22 = ILP_pat.addConstrs((
    gp.quicksum(
        x_pat[k, d-1]
        for k in range(len(P))
        ) <=
    gp.quicksum(
        x_pat[k, d]
        for k in range(len(P))
        )
    for d in range(2, D)),
    name="c22"
) # Constraint (22)

c23 = ILP_pat.addConstrs((
    gp.quicksum(
        a[k, j] * x_pat[k, d]
        for k in range(len(P))
        for d in range(D)
    ) == 1
    for j in range(n)),
    name="c23"
) # Constraint (23)

c24 = ILP_pat.addConstrs((
    gp.quicksum(
        y_pat[i, d1, d2]
        for d1 in range(D)
        for d2 in range(d1, D)
    ) == 1
    for i in range(m)),
    name="c24"
) # Constraint (24)

c25 = ILP_pat.addConstrs((
    gp.quicksum(
        T[i, j] * a[k, j] * x_pat[k, d]
        for k in range(len(P))
        for j in range(n)
    ) <=
    gp.quicksum(
        y_pat[i, d1, d2]
        for d2 in range(d, D)
        for d1 in range(d)
    )
    for i in range(m) for d in range(D)),
    name="c25"
) # Constraint (25)

# --- Constraints (26-27) --- #
# These constraints are embedded in the init of the variables

In [ ]:
# --- optimize that shit --- #
ILP_pat.optimize()
sol = ILP_pat.getJSONSolution()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: AMD Ryzen 7 4700U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Academic license 2773076 - for non-commercial use only - registered to km___@uga.edu
Optimize a model with 17 rows, 2511 columns and 1890 nonzeros (Min)
Model fingerprint: 0x179b51ed
Model has 405 linear objective coefficients
Model has 90 quadratic constraints
Variable types: 0 continuous, 2511 integer (2511 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 1e+00]
  Objective range  [6e+02, 8e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
  QRHS range       [1e+00, 1e+00]

Presolve added 21 rows and 0 columns
Presolve removed 0 rows and 382 columns
Presolve time: 0.12s
Presolved: 24176 rows, 10148 columns, 99365 nonzeros
Variable types: 0 contin

In [ ]:
all_vars = ILP_pat.getVars()
values = ILP_pat.getAttr("X", all_vars)
names = ILP_pat.getAttr("VarName", all_vars)

for name, val in zip(names, values):
    if val == 1:
        print(f"{name} = {val}")

In [ ]:
sol